# CSDE: Corrected Spatial Differential Expression

**CSDE** (`scviva.tl.csde`, `scviva.tl.CSDEAnalysis`) corrects for systematic errors introduced by automated spatial transcriptomics pipelines (cell mis-segmentation, mislabeling) before they propagate into false discoveries. It combines a large automated-annotation set with a small manually-validated subset via prediction-powered inference (PPI) to recover unbiased differential-expression estimates -- log-fold-changes, p-values, and BH-adjusted q-values -- with valid confidence intervals, for exactly two spatial populations of one cell type at a time (e.g. "inside tumour" vs. "outside tumour" macrophages).

**This notebook has not been executed in this environment (no GPU available). Run it once on a GPU-equipped machine before publishing rendered outputs.**

**Synthetic placeholder data.** CSDE's real inputs are a MERSCOPE/Xenium-style `SpatialData` zarr (fluorescence image, cell-boundary shapes, transcript points) whose `"table"` `AnnData` carries `cell_type`, `spatial_group`, `center_x`, `center_y` in `.obs` -- see `docs/user_guide/models/csde.md`. To keep this tutorial self-contained and reproducible without a network download, we generate placeholder spatial structure with `spatialdata`'s own synthetic test-data generator, `spatialdata.datasets.blobs()`, and a matching synthetic gene-count table built with plain `numpy`/`pandas`. **Substitute a real dataset meeting the requirements above (e.g. a MERSCOPE or Xenium `SpatialData` object with real `cell_type`/`spatial_group` annotations) before drawing any biological conclusions.**

## 1. Synthetic spatial dataset via `spatialdata.datasets.blobs()`

`spatialdata.datasets.blobs()` returns a small synthetic `SpatialData` object with the same *kind* of elements a real MERSCOPE/Xenium `SpatialData` zarr would have: a multi-channel fluorescence-like image (`blobs_image`), segmentation labels/shapes standing in for cell boundaries (`blobs_labels`, `blobs_circles`, `blobs_polygons`, `blobs_multipolygons`), simulated transcript detections (`blobs_points`), and a bundled `"table"` `AnnData`. We call it here just to illustrate that structure.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import spatialdata as sd

from scviva.tools.csde import CSDEAnalysis

rng = np.random.default_rng(0)

# Synthetic SpatialData object (image, cell-boundary shapes, transcript points, and a
# bundled placeholder `table`) standing in for a real MERSCOPE/Xenium `SpatialData` zarr.
sdata = sd.datasets.blobs(length=512, n_points=500, n_shapes=10)
sdata

`spatialdata.datasets.blobs()`'s bundled `"table"` `AnnData` is intentionally tiny (a handful of segmented regions, a few channel-intensity "genes", non-count values) -- it is meant to exercise `SpatialData` I/O, not to stand in for a real gene-expression panel. For a CSDE demo we need a cell-by-gene *count* table with `cell_type`, `spatial_group`, `center_x`, `center_y` in `.obs`, so we build that directly with `numpy`/`pandas`, positioning cells within the same image extent as `sdata["blobs_image"]` so it stays consistent with the synthetic spatial context above. On real data this table is `sdata["table"]` itself, produced by `spatialdata_io.xenium(...)` / `spatialdata_io.merscope(...)` plus a cell-typing step (e.g. scANVI) and a spatial-group split (e.g. a scVIVA niche assignment, or a distance-to-landmark threshold).

In [ ]:
image_length = sdata["blobs_image"].shape[-1]  # 512, matches `length` above

n_cells = 2000
n_genes = 50
gene_names = [f"gene_{i}" for i in range(n_genes)]

center_x = rng.uniform(0, image_length, size=n_cells)
center_y = rng.uniform(0, image_length, size=n_cells)
counts = rng.poisson(lam=5.0, size=(n_cells, n_genes)).astype(np.float32)

obs = pd.DataFrame(
    {"center_x": center_x, "center_y": center_y},
    index=[f"cell_{i}" for i in range(n_cells)],
)
var = pd.DataFrame(index=gene_names)
adata = ad.AnnData(X=counts, obs=obs, var=var)
adata

## 2. Derive `cell_type` and `spatial_group`

On real data, `cell_type` would come from an annotation step (e.g. scANVI/Leiden clustering + marker-based naming) and `spatial_group` from a niche assignment or a bounding-box/distance-to-landmark split of `center_x`/`center_y`. Here we assign both synthetically with `numpy.random`: two cell types, and a `spatial_group` split of the tissue into two halves along `center_x` ("region_A" vs. "region_B"). We then inject a true differential-expression signal into 10 genes for one cell type in `region_B`, so `test_differential_expression()` below has a real effect to recover.

In [ ]:
cell_type = rng.choice(["Macrophage", "Tcell"], size=n_cells, p=[0.4, 0.6])
adata.obs["cell_type"] = pd.Categorical(cell_type)

# spatial_group: simple bounding-box split of the tissue into two regions to compare
# (stand-in for e.g. a scVIVA niche assignment or a distance-to-landmark threshold).
midpoint = image_length / 2
adata.obs["spatial_group"] = np.where(
    adata.obs["center_x"].to_numpy() >= midpoint, "region_B", "region_A"
)
adata.obs["spatial_group"] = adata.obs["spatial_group"].astype("category")

# Inject a synthetic DE signal: bump expression of the first 10 genes in Macrophages
# inside region_B, so there is a real effect for CSDE to recover below.
de_genes = gene_names[:10]
target_mask = (
    (adata.obs["cell_type"] == "Macrophage").to_numpy()
    & (adata.obs["spatial_group"] == "region_B").to_numpy()
)
gene_idx = [adata.var_names.get_loc(g) for g in de_genes]
adata.X[np.ix_(np.where(target_mask)[0], gene_idx)] = rng.poisson(
    lam=15.0, size=(int(target_mask.sum()), len(gene_idx))
).astype(np.float32)

# CSDE compares two spatial populations of the *same* cell type; combine both obs columns
# into a single label so it can be passed as `pred_cell_pop_key`.
adata.obs["population"] = (
    adata.obs["cell_type"].astype(str) + "_" + adata.obs["spatial_group"].astype(str)
)
adata.obs[["cell_type", "spatial_group", "population", "center_x", "center_y"]].head()

## 3. Stand-in ground-truth subset

CSDE's real ground-truth subset comes from upstream CSDE's manual-validation workflow -- `scripts/export.py` renders a per-cell image panel (fluorescence crop + transcript dots + top-gene bar chart) for a small importance-sampled subset, and `scripts/annotate.py` (a Streamlit UI) lets a human mark each cell as correctly or incorrectly segmented/labeled, producing `annotations.json`. Neither step is ported into scviva-tools (see `docs/user_guide/models/csde.md`, "What's not (yet) in scviva-tools"). As a stand-in here, we take a random subsample of cells and treat the existing (synthetic) label as ground truth -- i.e. `is_correct=True` for every sampled cell, with a uniform `sampling_weight`. **On real data, replace this with an actually- validated `is_correct` column from upstream CSDE's annotation workflow** (optionally with non-uniform `sampling_weight` if the exported panel deliberately oversampled rare cell types).

In [ ]:
n_validate = 300
validate_idx = rng.choice(adata.n_obs, size=n_validate, replace=False)
adata_gt = adata[validate_idx].copy()

# Stand-in for real manual validation (upstream CSDE's export.py + annotate.py UI, not
# ported here): treat the existing automated label as ground truth. On real data, replace
# this with an actually-validated `is_correct` column from upstream CSDE's annotation
# workflow.
adata_gt.obs["is_correct"] = True
adata_gt.obs["sampling_weight"] = 1.0
adata_gt

## 4. Run CSDE

We compare `Macrophage` cells in `region_A` (reference) against `Macrophage` cells in `region_B` (target), using the base `CSDEAnalysis` constructor directly with plain `AnnData` objects and column names -- not `CSDEAnalysis.from_spatialdata()`, which expects a real upstream annotation directory (`config.json`, `metadata.csv`, `annotations.json`) that we don't have for this synthetic example. `adata_pred` is the full automated population; `adata_gt` is the manually-validated (here: stand-in) subset from above.

In [ ]:
cell_pop_a = "Macrophage_region_A"  # reference
cell_pop_b = "Macrophage_region_B"  # target

analysis = CSDEAnalysis(
    adata_pred=adata,
    adata_gt=adata_gt,
    pred_cell_pop_key="population",
    cell_pop_a=cell_pop_a,
    cell_pop_b=cell_pop_b,
    gt_key="is_correct",
)
analysis.fit(noise_model="poisson", optimizer="lbfgs")
results = analysis.test_differential_expression()
results.sort_values("p_value_adj").head(20)

## 5. Volcano plot

A simple volcano plot of the corrected log-fold-change against the BH-adjusted p-value, consistent with how other scviva-tools tutorials visualize DE results. The 10 genes with the injected synthetic signal (`gene_0`-`gene_9`) should stand out as the most significant, highest-|log-fold-change| points.

In [ ]:
neglog10_padj = -np.log10(results["p_value_adj"].clip(lower=1e-300))

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(results["log_fold_change"], neglog10_padj, s=12, alpha=0.6, color="tab:blue")
ax.axhline(-np.log10(0.05), color="grey", linestyle="--", linewidth=1, label="padj = 0.05")
ax.axvline(0, color="grey", linestyle="-", linewidth=0.5)
ax.set_xlabel("log fold change (Macrophage: region_B vs. region_A)")
ax.set_ylabel("-log10(adjusted p-value)")
ax.set_title("CSDE volcano plot (synthetic data)")
ax.legend()
fig.tight_layout()
fig

## Closing notes

This tutorial used entirely synthetic placeholder data (`spatialdata.datasets.blobs()` for spatial structure plus a hand-built `numpy`/`pandas` count table) so it runs without any external download and stays reproducible. For real use:

-   Replace the synthetic table with a real `SpatialData` `"table"` `AnnData` (e.g. from `spatialdata_io.xenium(...)` or `spatialdata_io.merscope(...)`) carrying real `cell_type`, `spatial_group`, `center_x`, `center_y` values in `.obs`.
-   Replace the stand-in ground-truth subset with a real manually-validated one, produced by upstream [YosefLab/CSDE](https://github.com/YosefLab/CSDE)'s `scripts/export.py` + `scripts/annotate.py`, then load it with `CSDEAnalysis.from_spatialdata()` instead of the base constructor used here.

See `docs/user_guide/models/csde.md` for a full summary of what's ported into scviva-tools (the PPI-based differential-expression step) and what is not (the panel-export and manual-validation UI), and the upstream [YosefLab/CSDE](https://github.com/YosefLab/CSDE) repository for the complete export/annotate/DE pipeline.